# 💾 Download Images under 20minutes instead of 4hours

As the original script is not optimized, I have updated it to speed up the download from ~4hours to 20 minutes. 
Below you can find the approach ffor directl use on Kaggle and code snippet ready for copy paste into your project.

**Authors:**
- Eric Orenstein (eorenstein@mbari.org)
- Lukas Picek (lukaspicek@gmail.com)


# 🏃 Run this to retrive the images on Kaggle


⚠️ There is not enough space on Kaggle to store the images in original resolution. In that case, you have to resize it before saving. Just set `resize = True` ⚠️

In [1]:
import os
import json
import time
import logging
import requests

from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def make_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=32, pool_maxsize=32)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def resize_image(file_name):
    try:
        img = Image.open(file_name)
        img.thumbnail((1280, 720), Image.LANCZOS)
        img.save(file_name)
    except Exception as e:
        print(f"Problem resizing {file_name}: {e}")


def download_img(args):
    name, url, outdir, resize = args
    file_name = os.path.join(outdir, name)
    os.makedirs(os.path.dirname(file_name), exist_ok=True)

    if os.path.exists(file_name):
        return ("exists", name)

    session = make_session()
    tmp_name = file_name + ".part"

    try:
        with session.get(url, stream=True, timeout=(10, 60)) as resp:
            resp.raise_for_status()

            with open(tmp_name, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 64):
                    if chunk:
                        f.write(chunk)

        os.replace(tmp_name, file_name)

        if resize:
            resize_image(file_name)

        return ("downloaded", name)

    except Exception as e:
        if os.path.exists(tmp_name):
            try:
                os.remove(tmp_name)
            except OSError:
                pass
        return ("failed", f"{name}: {e}")


def download_imgs(imgs, outdir=None, max_workers=16, resize=True):
    if outdir is None:
        outdir = os.path.join(os.getcwd(), "images")

    os.makedirs(outdir, exist_ok=True)

    args_list = [(name, url, outdir, resize) for name, url in imgs]

    downloaded = 0
    failed = 0
    existed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(download_img, args) for args in args_list]

        with tqdm(total=len(futures), desc="Downloading") as pbar:
            for future in as_completed(futures):
                status, info = future.result()

                if status == "downloaded":
                    downloaded += 1
                elif status == "exists":
                    existed += 1
                else:
                    failed += 1
                    print(f"Failed: {info}")

                pbar.update(1)
                pbar.set_postfix(downloaded=downloaded, exists=existed, failed=failed)

    print(f"\nDone. downloaded={downloaded}, exists={existed}, failed={failed}")

c:\Users\divya\anaconda3\envs\504FinalProject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = "train.json"
outpath = "kaggle/training-data"

logging.info(f"opening {dataset}")
with open(dataset, "r") as ff:
    dataset = json.load(ff)

ims = dataset["images"]
logging.info(f"retrieving {len(ims)} images")

ims = [(im["file_name"], im["coco_url"]) for im in ims]

download_imgs(ims, outdir=outpath, max_workers=16, resize=True)

Downloading:  36%|███▌      | 2885/8058 [00:03<00:05, 906.41it/s, downloaded=0, exists=2776, failed=109]

Failed: 8a1cfdce-09b1-4e72-ae3c-5a5e731378b6.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0566/01_32_19_01.png
Failed: 2365221d-8842-48f8-9749-d582477bd2fd.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0976/10_01_47_10.png
Failed: 3ceb5821-c194-4769-b89b-2eea73eeac4b.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0566/01_32_15_20.png
Failed: 4d5c96ec-ae52-49a9-863d-d4b7a00add08.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0536/03_34_18_27.png
Failed: 2e32fd92-5ec8-4136-a376-93d8f82d82d9.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0566/01_40_13_00.png
Failed: be9812f9-a6c3-4577-8dfb-fe889150e349.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricke

Downloading:  36%|███▌      | 2895/8058 [00:03<00:05, 906.41it/s, downloaded=0, exists=2776, failed=119]

Failed: ba4ecf45-d461-4538-8bb4-e603d24bb5c4.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0297/07_06_22_16.png
Failed: bb499ee9-ea19-47b2-9bd4-757cb86afc53.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0999/00_56_00_05.png
Failed: 927ecd29-692b-4fd8-93a5-cf8ed4ab058e.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0306/00_02_28_26.png
Failed: 67d55379-18ca-40ec-b9da-6aa7117e4e1a.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0295/03_30_45_28.png
Failed: 08fbd5dd-f1b0-41c7-a5be-627e7d7d1dad.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0540/00_05_05_07.png
Failed: 0c6a645d-d786-4d3e-8578-8cdbd77f6295.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricke

Downloading:  36%|███▌      | 2903/8058 [00:03<00:05, 906.41it/s, downloaded=0, exists=2776, failed=127]

...

Downloading:  36%|███▋      | 2935/8058 [00:06<00:05, 906.41it/s, downloaded=0, exists=2776, failed=159]

Failed: 47a788ca-7902-4852-ab1c-0265c4dbe273.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0568/01_13_01_15.png


Downloading:  36%|███▋      | 2936/8058 [00:07<00:05, 906.41it/s, downloaded=0, exists=2776, failed=160]

Failed: 9ddae789-f5a6-41d9-b912-bcf09ce693a9.png: 404 Client Error: Not Found for url: https://www.fathomnet.org/static/m3/staging/Doc-Ricketts/images/0999/01_01_43_19.png


Downloading:  36%|███▋      | 2936/8058 [00:34<01:00, 85.03it/s, downloaded=0, exists=2776, failed=160] 


In [ ]:
dataset = "eval.json"
outpath = "kaggle/test-data"

logging.info(f"opening {dataset}")
with open(dataset, "r") as ff:
    dataset = json.load(ff)

ims = dataset["images"]
logging.info(f"retrieving {len(ims)} images")

ims = [(im["file_name"], im["coco_url"]) for im in ims]

download_imgs(ims, outdir=outpath, max_workers=16, resize=True)